# Progressive pass — сравнение определений A / B / C

По `docs/metrics.md`:

- **A:** `progression >= 0.20`
- **B:** `progression >= 0.25` (текущая витрина)
- **C:** `progression >= 0.25` и `absolute_reduction >= 10` ярдов

Смотрим число передач, топ игроков/команд и зоны на финале WC2022 (`3869685`) и на всём турнире (только `PUBLISHABLE`).

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)

from src.db import connect

MATCH_ID = 3869685

In [ ]:
BASE_SQL = """
WITH base AS (
    SELECT
        e.canonical_match_id AS match_id,
        e.canonical_team_id AS team_id,
        e.canonical_player_id AS player_id,
        e.x AS start_x,
        e.y AS start_y,
        SQRT(POWER(120 - e.x, 2) + POWER(40 - e.y, 2)) AS distance_before,
        SQRT(POWER(120 - e.end_x, 2) + POWER(40 - e.end_y, 2)) AS distance_after,
        CASE
            WHEN SQRT(POWER(120 - e.x, 2) + POWER(40 - e.y, 2)) = 0 THEN NULL
            ELSE (
                SQRT(POWER(120 - e.x, 2) + POWER(40 - e.y, 2))
                - SQRT(POWER(120 - e.end_x, 2) + POWER(40 - e.end_y, 2))
            ) / SQRT(POWER(120 - e.x, 2) + POWER(40 - e.y, 2))
        END AS progression,
        SQRT(POWER(120 - e.x, 2) + POWER(40 - e.y, 2))
            - SQRT(POWER(120 - e.end_x, 2) + POWER(40 - e.end_y, 2)) AS absolute_reduction
    FROM core.events e
    JOIN monitoring.match_publication_status mps
      ON mps.canonical_match_id = e.canonical_match_id
     AND mps.publication_status = 'PUBLISHABLE'
    WHERE e.event_type = 'Pass'
      AND e.x IS NOT NULL AND e.y IS NOT NULL
      AND e.end_x IS NOT NULL AND e.end_y IS NOT NULL
)
SELECT * FROM base
"""

with connect() as conn:
    passes = pd.read_sql(BASE_SQL, conn)

passes["A"] = passes["progression"] >= 0.20
passes["B"] = passes["progression"] >= 0.25
passes["C"] = (passes["progression"] >= 0.25) & (passes["absolute_reduction"] >= 10)

summary = pd.DataFrame(
    {
        "definition": ["A", "B", "C"],
        "tournament": [passes["A"].sum(), passes["B"].sum(), passes["C"].sum()],
        "final": [
            passes.loc[passes.match_id == MATCH_ID, "A"].sum(),
            passes.loc[passes.match_id == MATCH_ID, "B"].sum(),
            passes.loc[passes.match_id == MATCH_ID, "C"].sum(),
        ],
    }
)
display(summary)
assert summary.loc[0, "tournament"] >= summary.loc[1, "tournament"] >= summary.loc[2, "tournament"]

In [ ]:
def top_n(mask_col: str, group_col: str, n: int = 10) -> pd.DataFrame:
    return (
        passes.loc[passes[mask_col]]
        .groupby(group_col)
        .size()
        .sort_values(ascending=False)
        .head(n)
        .rename("passes")
        .reset_index()
    )

print("Top players A/B/C")
for d in "ABC":
    print(d)
    display(top_n(d, "player_id"))

print("Top teams A/B/C")
for d in "ABC":
    print(d)
    display(top_n(d, "team_id"))

In [ ]:
def zone(x: float, y: float) -> str:
    # coarse thirds for a quick distribution check
    third = "def" if x < 40 else ("mid" if x < 80 else "att")
    lane = "left" if y < 26.67 else ("center" if y < 53.33 else "right")
    return f"{third}_{lane}"

passes["start_zone"] = [
    zone(x, y) for x, y in zip(passes["start_x"], passes["start_y"])
]

zone_cmp = pd.DataFrame({
    d: passes.loc[passes[d], "start_zone"].value_counts()
    for d in "ABC"
}).fillna(0).astype(int).sort_index()
display(zone_cmp)

print(
    "Итог эксперимента: оставляем определение B в marts.progressive_passes "
    "(баланс объёма vs шум рядом с воротами; C остаётся кандидатом)."
)